# Статическая среда: FEM, SVD-SNO и FNO

Обе модели используют одинаковые static-данные и test-частоты. В каждой полосе — одна среда, 1000 частот и по восемь направлений на частоту; train/validation/test: 5360/1360/1280 полей. Для low и high обучены отдельные модели.

Рисунки показывают Re, Im, Abs и ошибки соответствующих компонент. Пример выбирается по ближайшей к медиане test-ошибке SNO; FNO не влияет на отбор. Общие цветовые шкалы позволяют сопоставлять поля и ошибки. Исторический JAX-эксперимент статьи и старый пилот с переменными средами относятся к другим данным.

Этот notebook проверяет сохранённые модели и отображает результаты. Обучение и FEM-решатель не запускаются.


In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'neural_operator').is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
PAPER = ROOT / 'paper' / 'svd_sno_revision_2026-11-06'
if str(PAPER) not in sys.path:
    sys.path.insert(0, str(PAPER))
import json
from IPython.display import Image, display
from neural_operator.plotting import display_table
from prepare_static_comparison import verify_comparison


In [ ]:
rows = []
for band in ('low', 'high'):
    dataset, index, predictions, record = verify_comparison(band)
    print(f'{band}: проверены fingerprint, сетка, все 1280 test-ошибок и совпадение checkpoint.')
    for method, model in record['models'].items():
        errors = model['test_complex_consistent_fem_l2']
        rows.append({'band': band, 'method': method,
                     'mean_%': 100*errors['mean'], 'median_%': 100*errors['median'],
                     'p95_%': 100*errors['p95'], 'max_%': 100*errors['max'],
                     'sample_index': index, 'sample_%': model['sample_error_percent'],
                     'epochs': model['completed_epochs'], 'best_epoch': model['best_epoch'],
                     'training_device': model['config']['device']})
    del dataset, predictions
display_table(rows)


## Re / Im / Abs на одинаковых тестовых примерах

Строки: Re, Im, Abs. Столбцы: FEM, SVD-SNO, ошибка SVD-SNO, FNO, ошибка FNO. Ошибка Abs равна `abs(abs(pred)-abs(FEM))`; комплексная FEM L2 метрика в таблице дополнительно учитывает фазу.


In [ ]:
for band in ('low', 'high'):
    path = PAPER / 'images' / f'prediction_static_{band}_comparison_components.png'
    if not path.exists():
        raise FileNotFoundError(f'Постройте рисунки: python {PAPER / "prepare_static_comparison.py"}')
    display(Image(filename=str(path)))


SNO обучен на CPU, FNO — на MPS. Архитектуры имеют разное число параметров; общими являются данные, split и основные настройки оптимизации. По этим результатам оцениваются конкретные сохранённые модели; прямое сравнение времени обучения не проводится.

[Методичка](../docs/svd_sno_guide.md) · [Отчёт SNO](../randomized/RESULTS.md) · [Отчёт FNO](../randomized/FNO_RESULTS.md). В `paper/svd_sno_revision_2026-11-06/images/prediction_static_comparison_provenance.json` сохранены источники, хеши, конфигурации и проверки.
